# Notebook 1: Basic One-Compartment PK/PD Simulation: From Plasma Concentration to Drug Effect

This notebook is the first foundational module of the clinical pharmacy PK/PD interactive simulation platform.

Using the classic one-compartment model, this notebook helps you build the basic PK/PD reasoning framework used in clinical pharmacy:

- How does dose affect plasma drug concentration?
- How do concentration-time curves differ after IV bolus dosing and oral dosing?
- How do the volume of distribution (Vd) and clearance (CL) affect plasma concentration?
- How are half-life (t1/2) and AUC calculated?
- How can plasma concentration be translated into drug effect?
- Why does increasing the dose not always mean greater clinical benefit?

The core logic of this notebook is:

$$
Dose \rightarrow Concentration(t) \rightarrow Exposure \rightarrow Effect
$$

In other words, drug dose is not directly equal to drug effect.  
The dose first determines the drug concentration in the body, and the concentration is then converted into drug effect through a pharmacodynamic model.

## 1. Learning Objectives

After completing this notebook, you should be able to:

1. Explain key one-compartment PK parameters, including Dose, Vd, CL, k, and t1/2.
2. Compare IV bolus and oral dosing concentration-time profiles.
3. Describe how F, ka, Vd, and CL influence concentration, exposure, Cmax, Tmax, AUC, and half-life.
4. Use MEC and MTC to interpret preliminary efficacy and toxicity risks.
5. Link plasma concentration to drug effect using a basic Emax PK/PD model.

## 2. Basic Concept of the One-Compartment Model

The one-compartment model is the simplest and most fundamental pharmacokinetic model.

It simplifies the human body as a "well-mixed container." After a drug enters the body, it is assumed to distribute rapidly into an apparent space. The size of this space is called the apparent volume of distribution:

$$
V_d
$$

The body's ability to eliminate the drug is called clearance:

$$
CL
$$

The elimination rate constant is:

$$
k = \frac{CL}{V_d}
$$

The half-life is:

$$
t_{1/2} = \frac{0.693}{k}
$$

It can also be written as:

$$
t_{1/2} = \frac{0.693 \times V_d}{CL}
$$

Therefore, half-life is not an isolated parameter. It is jointly determined by Vd and CL.

## 3. One-Compartment Model for IV Bolus Dosing

For IV bolus dosing, the drug enters the systemic circulation directly, so there is no absorption process.

After dosing, the plasma concentration can be expressed as:

$$
C(t) = \frac{Dose}{V_d} \cdot e^{-kt}
$$

where:

$$
k = \frac{CL}{V_d}
$$

After IV bolus dosing, the maximum concentration usually occurs immediately after dosing:

$$
C_{max} = C_0 = \frac{Dose}{V_d}
$$

The AUC after a single IV dose is:

$$
AUC = \frac{Dose}{CL}
$$

Variable definitions:

| Symbol | Meaning | Common unit |
|---|---|---|
| Dose | Administered dose | mg |
| Vd | Apparent volume of distribution | L |
| CL | Clearance | L/h |
| k | Elimination rate constant | 1/h |
| t1/2 | Half-life | h |
| C(t) | Plasma concentration at time t | mg/L |
| Cmax | Peak concentration | mg/L |
| AUC | Area under the concentration-time curve | mg·h/L |

Key points:

- A larger Dose leads to a higher initial plasma concentration.
- A larger Vd means the same dose is “diluted” into a larger apparent space, resulting in a lower initial concentration.
- A larger CL means faster drug elimination and a faster decline in plasma concentration.
- k is determined jointly by CL and Vd.
- AUC is directly proportional to Dose and inversely proportional to CL.

## 4. One-Compartment Model for Oral Dosing

Oral dosing differs from IV dosing. The drug must first be absorbed from the gastrointestinal tract into the systemic circulation, so two additional important parameters are needed:

| Parameter | Meaning |
|---|---|
| F | Bioavailability, the fraction that reaches the systemic circulation |
| ka | Absorption rate constant, describing how fast absorption occurs |

For a one-compartment model with first-order absorption and first-order elimination, the plasma concentration after oral dosing can be expressed as:

$$
C(t) = \frac{F \cdot Dose \cdot k_a}{V_d(k_a-k)}
\left(e^{-kt} - e^{-k_a t}\right)
$$

where:

$$
k = \frac{CL}{V_d}
$$

The AUC after oral dosing is:

$$
AUC = \frac{F \cdot Dose}{CL}
$$

Compared with IV dosing, oral dosing has the following features:

- The concentration does not start from the highest point and then decline; instead, it rises first and then falls.
- Because there is an absorption process, Tmax is present.
- Cmax is jointly affected by Dose, F, Vd, CL, and ka.
- A lower F reduces systemic exposure, measured by AUC.
- A larger ka means faster absorption, and Tmax usually occurs earlier.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ipywidgets import interact, FloatSlider, IntSlider, Dropdown

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except Exception:
    pass

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True


def one_compartment_iv_bolus(t, dose_mg, vd_l, cl_l_h):
    """
    One-compartment IV bolus model.
    """
    k_elim = cl_l_h / vd_l
    concentration = (dose_mg / vd_l) * np.exp(-k_elim * t)
    auc = dose_mg / cl_l_h
    half_life = np.log(2) / k_elim

    return concentration, k_elim, half_life, auc


def one_compartment_oral(t, dose_mg, vd_l, cl_l_h, ka_h, bioavailability):
    """
    One-compartment oral dosing model with first-order absorption and elimination.
    """
    k_elim = cl_l_h / vd_l

    if np.isclose(ka_h, k_elim):
        concentration = (
            bioavailability
            * dose_mg
            / vd_l
            * k_elim
            * t
            * np.exp(-k_elim * t)
        )
    else:
        concentration = (
            bioavailability
            * dose_mg
            * ka_h
            / (vd_l * (ka_h - k_elim))
            * (np.exp(-k_elim * t) - np.exp(-ka_h * t))
        )

    concentration = np.maximum(concentration, 0)
    auc = bioavailability * dose_mg / cl_l_h
    half_life = np.log(2) / k_elim

    return concentration, k_elim, half_life, auc


def calculate_pk_metrics(t, concentration, auc, half_life):
    """
    Calculate basic PK metrics.
    """
    cmax = np.max(concentration)
    tmax = t[np.argmax(concentration)]

    metrics = {
        "Cmax": cmax,
        "Tmax": tmax,
        "AUC": auc,
        "Half-life": half_life
    }

    return metrics

## 5. Interactive Simulation 1: Comparing IV and Oral Dosing

In the simulation below, you can choose between two routes of administration:

- IV bolus: intravenous bolus dosing
- Oral: oral dosing

Please observe:

- After IV dosing, does the concentration begin to decline from the highest point?
- After oral dosing, does the concentration rise first and then fall?
- When F changes, how does the AUC of oral dosing change?
- When ka changes, how do Tmax and Cmax change?
- When CL changes, how do the elimination rate and AUC change?

In [ ]:
def plot_basic_pk_model(
    route="Oral",
    dose_mg=500,
    vd_l=50,
    cl_l_h=5,
    ka_h=1.2,
    bioavailability=0.8,
    t_end_h=24
):
    t = np.linspace(0, t_end_h, 1000)

    if route == "IV bolus":
        concentration, k_elim, half_life, auc = one_compartment_iv_bolus(
            t=t,
            dose_mg=dose_mg,
            vd_l=vd_l,
            cl_l_h=cl_l_h
        )
    else:
        concentration, k_elim, half_life, auc = one_compartment_oral(
            t=t,
            dose_mg=dose_mg,
            vd_l=vd_l,
            cl_l_h=cl_l_h,
            ka_h=ka_h,
            bioavailability=bioavailability
        )

    metrics = calculate_pk_metrics(
        t=t,
        concentration=concentration,
        auc=auc,
        half_life=half_life
    )

    fig, ax = plt.subplots()
    ax.plot(t, concentration, linewidth=2, label=f"{route}")

    ax.scatter(metrics["Tmax"], metrics["Cmax"], zorder=5)
    ax.text(
        metrics["Tmax"],
        metrics["Cmax"],
        f"  Cmax={metrics['Cmax']:.2f}, Tmax={metrics['Tmax']:.2f} h",
        va="center"
    )

    ax.set_title("One-Compartment PK Model")
    ax.set_xlabel("Time (h)")
    ax.set_ylabel("Concentration (mg/L)")
    ax.legend()
    plt.show()

    summary = pd.DataFrame({
        "Parameter": [
            "Route",
            "Dose",
            "Vd",
            "CL",
            "ka",
            "Bioavailability",
            "Elimination rate constant",
            "Half-life",
            "Cmax",
            "Tmax",
            "AUC"
        ],
        "Value": [
            route,
            f"{dose_mg:.1f} mg",
            f"{vd_l:.1f} L",
            f"{cl_l_h:.1f} L/h",
            f"{ka_h:.2f} 1/h",
            f"{bioavailability:.2f}",
            f"{k_elim:.4f} 1/h",
            f"{metrics['Half-life']:.2f} h",
            f"{metrics['Cmax']:.2f} mg/L",
            f"{metrics['Tmax']:.2f} h",
            f"{metrics['AUC']:.2f} mg*h/L"
        ]
    })

    display(summary)


interact(
    plot_basic_pk_model,
    route=Dropdown(
        options=["IV bolus", "Oral"],
        value="Oral",
        description="Route"
    ),
    dose_mg=FloatSlider(value=500, min=100, max=2000, step=100, description="Dose"),
    vd_l=FloatSlider(value=50, min=10, max=150, step=5, description="Vd"),
    cl_l_h=FloatSlider(value=5, min=1, max=20, step=1, description="CL"),
    ka_h=FloatSlider(value=1.2, min=0.1, max=5, step=0.1, description="ka"),
    bioavailability=FloatSlider(value=0.8, min=0.1, max=1.0, step=0.05, description="F"),
    t_end_h=FloatSlider(value=24, min=6, max=72, step=6, description="Time")
);

interactive(children=(Dropdown(description='Route', index=1, options=('IV bolus', 'Oral'), value='Oral'), Floa…

## 6. Observation Task 1: Effect of Route of Administration on the Concentration Curve

Please complete the following steps:

### Task A: IV dosing

Settings:

- Route = IV bolus
- Dose = 500 mg
- Vd = 50 L
- CL = 5 L/h
- Time = 24 h

Observe:

- Does the concentration begin to decline from the highest point?
- Is Tmax close to 0?
- Is Cmax approximately equal to Dose / Vd?
- Is AUC equal to Dose / CL?

### Task B: Oral dosing

Settings:

- Route = Oral
- Dose = 500 mg
- Vd = 50 L
- CL = 5 L/h
- ka = 1.2 1/h
- F = 0.8
- Time = 24 h

Observe:

- Does the concentration rise first and then fall?
- Is Tmax greater than 0?
- Is AUC lower than that after IV dosing at the same dose?
- Why is the AUC after oral dosing related to F?

### Task C: Change the absorption rate

Keep the other parameters unchanged and set ka to:

- 0.3 1/h
- 1.2 1/h
- 3.0 1/h

Observe:

- When ka increases, does Tmax occur earlier or later?
- When ka increases, does Cmax usually increase?
- When ka is very small, does the concentration curve become flatter?

## 7. Clinical Meaning of Half-Life and AUC

Half-life is the time required for the plasma concentration to decrease by half.

For a first-order elimination process:

- After 1 half-life, about 50% of the concentration remains.
- After 2 half-lives, about 25% of the concentration remains.
- After 3 half-lives, about 12.5% of the concentration remains.
- After 4 half-lives, about 6.25% of the concentration remains.
- After 5 half-lives, about 3.125% of the concentration remains.

AUC represents drug exposure.

For IV dosing:

$$
AUC = \frac{Dose}{CL}
$$

For oral dosing:

$$
AUC = \frac{F \cdot Dose}{CL}
$$

Therefore:

- When Dose increases, AUC increases.
- When CL increases, AUC decreases.
- For oral dosing, when F decreases, AUC decreases.
- When renal function or hepatic function decreases, CL may decrease; as a result, AUC may increase and toxicity risk may increase.

In [ ]:
def plot_iv_and_oral_comparison(
    dose_mg=500,
    vd_l=50,
    cl_l_h=5,
    ka_h=1.2,
    bioavailability=0.8,
    t_end_h=24
):
    t = np.linspace(0, t_end_h, 1000)

    conc_iv, k_elim, half_life_iv, auc_iv = one_compartment_iv_bolus(
        t=t,
        dose_mg=dose_mg,
        vd_l=vd_l,
        cl_l_h=cl_l_h
    )

    conc_oral, _, half_life_oral, auc_oral = one_compartment_oral(
        t=t,
        dose_mg=dose_mg,
        vd_l=vd_l,
        cl_l_h=cl_l_h,
        ka_h=ka_h,
        bioavailability=bioavailability
    )

    metrics_iv = calculate_pk_metrics(t, conc_iv, auc_iv, half_life_iv)
    metrics_oral = calculate_pk_metrics(t, conc_oral, auc_oral, half_life_oral)

    fig, ax = plt.subplots()
    ax.plot(t, conc_iv, linewidth=2, label="IV bolus")
    ax.plot(t, conc_oral, linewidth=2, label="Oral")

    ax.set_title("IV Bolus vs Oral Dosing")
    ax.set_xlabel("Time (h)")
    ax.set_ylabel("Concentration (mg/L)")
    ax.legend()
    plt.show()

    summary = pd.DataFrame({
        "Metric": ["Cmax", "Tmax", "AUC", "Half-life"],
        "IV bolus": [
            f"{metrics_iv['Cmax']:.2f} mg/L",
            f"{metrics_iv['Tmax']:.2f} h",
            f"{metrics_iv['AUC']:.2f} mg*h/L",
            f"{metrics_iv['Half-life']:.2f} h"
        ],
        "Oral": [
            f"{metrics_oral['Cmax']:.2f} mg/L",
            f"{metrics_oral['Tmax']:.2f} h",
            f"{metrics_oral['AUC']:.2f} mg*h/L",
            f"{metrics_oral['Half-life']:.2f} h"
        ]
    })

    display(summary)


interact(
    plot_iv_and_oral_comparison,
    dose_mg=FloatSlider(value=500, min=100, max=2000, step=100, description="Dose"),
    vd_l=FloatSlider(value=50, min=10, max=150, step=5, description="Vd"),
    cl_l_h=FloatSlider(value=5, min=1, max=20, step=1, description="CL"),
    ka_h=FloatSlider(value=1.2, min=0.1, max=5, step=0.1, description="ka"),
    bioavailability=FloatSlider(value=0.8, min=0.1, max=1.0, step=0.05, description="F"),
    t_end_h=FloatSlider(value=24, min=6, max=72, step=6, description="Time")
);

interactive(children=(FloatSlider(value=500.0, description='Dose', max=2000.0, min=100.0, step=100.0), FloatSl…

## 8. Observation Task 2: Comparing IV and Oral Dosing

Compare the two curves at the same dose in the figure above.

Think about the following questions:

1. Why does IV dosing have no absorption phase?
2. Why is Tmax later after oral dosing than after IV dosing?
3. Why is the AUC after oral dosing usually lower than that after IV dosing?
4. If F = 1, are the AUC values for oral dosing and IV dosing the same?
5. If ka is very small, what happens to the oral concentration curve?
6. If CL decreases, how do the AUC values for IV and oral dosing change?

## 9. Therapeutic Window: From Concentration to Clinical Judgment

In clinical pharmacy, we care not only about the plasma concentration itself, but also whether the concentration is within an appropriate range.

Two simplified concepts can help with this interpretation:

| Concept | Meaning |
|---|---|
| MEC | Minimum Effective Concentration |
| MTC | Minimum Toxic Concentration |

When the plasma concentration is below MEC, efficacy may be insufficient.

When the plasma concentration is above MTC, toxicity risk may increase.

When the plasma concentration is between MEC and MTC, it can be considered within the therapeutic window:

$$
MEC \leq C(t) \leq MTC
$$

Please note that MEC and MTC in this notebook are used only for teaching simulation. Real clinical judgment must also consider the drug, indication, patient organ function, concomitant medications, pathogen susceptibility, and treatment response.

In [ ]:
def calculate_time_in_ranges(t, concentration, mec, mtc):
    """
    Calculate time below MEC, within therapeutic window, and above MTC.
    """
    dt = t[1] - t[0]

    time_below_mec = np.sum(concentration < mec) * dt
    time_within_window = np.sum(
        (concentration >= mec) & (concentration <= mtc)
    ) * dt
    time_above_mtc = np.sum(concentration > mtc) * dt

    return time_below_mec, time_within_window, time_above_mtc


def plot_therapeutic_window(
    route="Oral",
    dose_mg=500,
    vd_l=50,
    cl_l_h=5,
    ka_h=1.2,
    bioavailability=0.8,
    mec=2,
    mtc=12,
    t_end_h=24
):
    t = np.linspace(0, t_end_h, 1000)

    if route == "IV bolus":
        concentration, k_elim, half_life, auc = one_compartment_iv_bolus(
            t=t,
            dose_mg=dose_mg,
            vd_l=vd_l,
            cl_l_h=cl_l_h
        )
    else:
        concentration, k_elim, half_life, auc = one_compartment_oral(
            t=t,
            dose_mg=dose_mg,
            vd_l=vd_l,
            cl_l_h=cl_l_h,
            ka_h=ka_h,
            bioavailability=bioavailability
        )

    metrics = calculate_pk_metrics(t, concentration, auc, half_life)

    time_below_mec, time_within_window, time_above_mtc = calculate_time_in_ranges(
        t=t,
        concentration=concentration,
        mec=mec,
        mtc=mtc
    )

    fig, ax = plt.subplots()
    ax.plot(t, concentration, linewidth=2, label=f"{route}")
    ax.axhline(mec, linestyle="--", label=f"MEC = {mec:.1f} mg/L")
    ax.axhline(mtc, linestyle="--", label=f"MTC = {mtc:.1f} mg/L")
    ax.fill_between(t, mec, mtc, alpha=0.15, label="Therapeutic window")

    ax.set_title("Concentration and Therapeutic Window")
    ax.set_xlabel("Time (h)")
    ax.set_ylabel("Concentration (mg/L)")
    ax.legend()
    plt.show()

    summary = pd.DataFrame({
        "Metric": [
            "Cmax",
            "Tmax",
            "AUC",
            "Half-life",
            "Time below MEC",
            "Time within therapeutic window",
            "Time above MTC"
        ],
        "Value": [
            f"{metrics['Cmax']:.2f} mg/L",
            f"{metrics['Tmax']:.2f} h",
            f"{metrics['AUC']:.2f} mg*h/L",
            f"{metrics['Half-life']:.2f} h",
            f"{time_below_mec:.2f} h",
            f"{time_within_window:.2f} h",
            f"{time_above_mtc:.2f} h"
        ]
    })

    display(summary)


interact(
    plot_therapeutic_window,
    route=Dropdown(
        options=["IV bolus", "Oral"],
        value="Oral",
        description="Route"
    ),
    dose_mg=FloatSlider(value=500, min=100, max=2000, step=100, description="Dose"),
    vd_l=FloatSlider(value=50, min=10, max=150, step=5, description="Vd"),
    cl_l_h=FloatSlider(value=5, min=1, max=20, step=1, description="CL"),
    ka_h=FloatSlider(value=1.2, min=0.1, max=5, step=0.1, description="ka"),
    bioavailability=FloatSlider(value=0.8, min=0.1, max=1.0, step=0.05, description="F"),
    mec=FloatSlider(value=2, min=0.5, max=10, step=0.5, description="MEC"),
    mtc=FloatSlider(value=12, min=5, max=30, step=1, description="MTC"),
    t_end_h=FloatSlider(value=24, min=6, max=72, step=6, description="Time")
);

interactive(children=(Dropdown(description='Route', index=1, options=('IV bolus', 'Oral'), value='Oral'), Floa…

## 10. Observation Task 3: Therapeutic Window and Efficacy/Toxicity Risk

Please complete the following steps:

### Task A: Standard oral dosing

Settings:

- Route = Oral
- Dose = 500 mg
- Vd = 50 L
- CL = 5 L/h
- ka = 1.2 1/h
- F = 0.8
- MEC = 2 mg/L
- MTC = 12 mg/L

Record:

- Cmax
- Tmax
- AUC
- Time below MEC
- Time within the therapeutic window
- Time above MTC

### Task B: Increase the dose

Change Dose to 1000 mg.

Observe:

- Does Cmax increase?
- Does AUC increase?
- Does Time above MTC increase?
- Is increasing the dose always safer?

### Task C: Decrease clearance

Change CL to 2 L/h.

Observe:

- Does AUC increase?
- Is half-life prolonged?
- Does Time above MTC increase?
- Which clinical patient populations could this situation simulate?

Hint: decreased renal function, decreased hepatic function, or drug-drug interactions may all lead to reduced clearance.

## 11. From PK to PD: The Emax Model

PK focuses on:

> How does the body handle the drug?

PD focuses on:

> How does the drug produce its effect?

In clinical pharmacy, we need to know not only the plasma concentration, but also how much pharmacologic effect that concentration can produce.

This notebook uses the most common Emax model to describe the relationship between concentration and effect:

$$
Effect = E_0 + \frac{E_{max} \cdot C^\gamma}{EC_{50}^{\gamma} + C^\gamma}
$$

where:

| Parameter | Meaning |
|---|---|
| E0 | Baseline effect |
| Emax | Maximum drug effect |
| EC50 | Concentration that produces 50% of the maximum effect |
| gamma | Hill coefficient, which determines the steepness of the curve |
| C | Plasma concentration |

When the concentration is low, increasing the concentration may produce a clear increase in effect.

When the concentration is high, the effect gradually approaches a plateau. At this point, further dose increases may provide limited additional efficacy, while toxicity risk may continue to increase.

In [ ]:
def emax_effect(concentration, e0, emax, ec50, gamma):
    """
    Emax pharmacodynamic model.
    """
    concentration = np.asarray(concentration)

    effect = e0 + (
        emax * concentration**gamma
    ) / (
        ec50**gamma + concentration**gamma
    )

    return effect


def plot_emax_curve(
    e0=0,
    emax=100,
    ec50=5,
    gamma=1.5,
    c_max=30
):
    c = np.linspace(0, c_max, 500)

    effect = emax_effect(
        concentration=c,
        e0=e0,
        emax=emax,
        ec50=ec50,
        gamma=gamma
    )

    effect_at_ec50 = emax_effect(
        concentration=ec50,
        e0=e0,
        emax=emax,
        ec50=ec50,
        gamma=gamma
    )

    fig, ax = plt.subplots()
    ax.plot(c, effect, linewidth=2, label="Effect")
    ax.scatter(ec50, effect_at_ec50, zorder=5)
    ax.axvline(ec50, linestyle="--", label=f"EC50 = {ec50:.2f} mg/L")
    ax.axhline(effect_at_ec50, linestyle=":", label="Effect at EC50")

    ax.set_title("Emax Concentration-Effect Relationship")
    ax.set_xlabel("Concentration (mg/L)")
    ax.set_ylabel("Effect (%)")
    ax.set_ylim(0, e0 + emax * 1.1)
    ax.legend()
    plt.show()


interact(
    plot_emax_curve,
    e0=FloatSlider(value=0, min=0, max=50, step=5, description="E0"),
    emax=FloatSlider(value=100, min=20, max=150, step=10, description="Emax"),
    ec50=FloatSlider(value=5, min=0.5, max=20, step=0.5, description="EC50"),
    gamma=FloatSlider(value=1.5, min=0.5, max=5, step=0.5, description="Hill"),
    c_max=FloatSlider(value=30, min=5, max=80, step=5, description="C max")
);

interactive(children=(FloatSlider(value=0.0, description='E0', max=50.0, step=5.0), FloatSlider(value=100.0, d…

## 12. Observation Task 4: Effects of PD Parameters on the Effect Curve

Please complete the following steps:

### Task A: Change EC50

Settings:

- Emax = 100
- EC50 = 5 mg/L
- Hill = 1.5

Then change EC50 to 10 mg/L.

Observe:

- Does the curve shift to the right?
- Is the drug effect lower at the same concentration?
- Does a larger EC50 represent increased or decreased drug sensitivity?

### Task B: Change the Hill coefficient

Change Hill from 1.5 to 4.

Observe:

- Does the curve become steeper?
- Can a small change in concentration lead to a marked change in effect?
- Might dose adjustment need to be more cautious for this type of drug?

### Task C: Change Emax

Change Emax from 100 to 60.

Observe:

- Does the maximum effect decrease?
- Even when the concentration is very high, is the effect still unable to exceed the new Emax?

## 13. Integrated Simulation: From Dose to Plasma Concentration to Drug Effect

Now we connect PK and PD.

The complete process is:

$$
Dose \rightarrow C(t) \rightarrow Effect(t)
$$

This simulation displays all of the following:

- Plasma concentration-time curve
- Therapeutic window
- Drug effect-time curve

You can choose IV dosing or oral dosing and adjust both PK and PD parameters.

In [ ]:
def plot_integrated_pkpd(
    route="Oral",
    dose_mg=500,
    vd_l=50,
    cl_l_h=5,
    ka_h=1.2,
    bioavailability=0.8,
    mec=2,
    mtc=12,
    emax=100,
    ec50=5,
    gamma=1.5,
    t_end_h=24
):
    t = np.linspace(0, t_end_h, 1000)

    if route == "IV bolus":
        concentration, k_elim, half_life, auc = one_compartment_iv_bolus(
            t=t,
            dose_mg=dose_mg,
            vd_l=vd_l,
            cl_l_h=cl_l_h
        )
    else:
        concentration, k_elim, half_life, auc = one_compartment_oral(
            t=t,
            dose_mg=dose_mg,
            vd_l=vd_l,
            cl_l_h=cl_l_h,
            ka_h=ka_h,
            bioavailability=bioavailability
        )

    effect = emax_effect(
        concentration=concentration,
        e0=0,
        emax=emax,
        ec50=ec50,
        gamma=gamma
    )

    metrics = calculate_pk_metrics(t, concentration, auc, half_life)

    time_below_mec, time_within_window, time_above_mtc = calculate_time_in_ranges(
        t=t,
        concentration=concentration,
        mec=mec,
        mtc=mtc
    )

    max_effect = np.max(effect)
    time_max_effect = t[np.argmax(effect)]
    average_effect = np.trapz(effect, t) / (t[-1] - t[0])

    fig, ax1 = plt.subplots(figsize=(10, 6))

    ax1.plot(t, concentration, linewidth=2, label="Concentration")
    ax1.axhline(mec, linestyle="--", label=f"MEC = {mec:.1f} mg/L")
    ax1.axhline(mtc, linestyle="--", label=f"MTC = {mtc:.1f} mg/L")
    ax1.fill_between(t, mec, mtc, alpha=0.15, label="Therapeutic window")

    ax1.set_xlabel("Time (h)")
    ax1.set_ylabel("Concentration (mg/L)")

    ax2 = ax1.twinx()
    ax2.plot(t, effect, linewidth=2, linestyle="-.", label="Effect")
    ax2.set_ylabel("Effect (%)")

    lines_1, labels_1 = ax1.get_legend_handles_labels()
    lines_2, labels_2 = ax2.get_legend_handles_labels()
    ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc="upper right")

    ax1.set_title("Integrated PK/PD Simulation")
    plt.show()

    summary = pd.DataFrame({
        "Metric": [
            "Route",
            "Cmax",
            "Tmax",
            "AUC",
            "Half-life",
            "Maximum effect",
            "Time of maximum effect",
            "Average effect",
            "Time below MEC",
            "Time within therapeutic window",
            "Time above MTC"
        ],
        "Value": [
            route,
            f"{metrics['Cmax']:.2f} mg/L",
            f"{metrics['Tmax']:.2f} h",
            f"{metrics['AUC']:.2f} mg*h/L",
            f"{metrics['Half-life']:.2f} h",
            f"{max_effect:.2f}%",
            f"{time_max_effect:.2f} h",
            f"{average_effect:.2f}%",
            f"{time_below_mec:.2f} h",
            f"{time_within_window:.2f} h",
            f"{time_above_mtc:.2f} h"
        ]
    })

    display(summary)


interact(
    plot_integrated_pkpd,
    route=Dropdown(
        options=["IV bolus", "Oral"],
        value="Oral",
        description="Route"
    ),
    dose_mg=FloatSlider(value=500, min=100, max=2000, step=100, description="Dose"),
    vd_l=FloatSlider(value=50, min=10, max=150, step=5, description="Vd"),
    cl_l_h=FloatSlider(value=5, min=1, max=20, step=1, description="CL"),
    ka_h=FloatSlider(value=1.2, min=0.1, max=5, step=0.1, description="ka"),
    bioavailability=FloatSlider(value=0.8, min=0.1, max=1.0, step=0.05, description="F"),
    mec=FloatSlider(value=2, min=0.5, max=10, step=0.5, description="MEC"),
    mtc=FloatSlider(value=12, min=5, max=30, step=1, description="MTC"),
    emax=FloatSlider(value=100, min=20, max=150, step=10, description="Emax"),
    ec50=FloatSlider(value=5, min=0.5, max=20, step=0.5, description="EC50"),
    gamma=FloatSlider(value=1.5, min=0.5, max=5, step=0.5, description="Hill"),
    t_end_h=FloatSlider(value=24, min=6, max=72, step=6, description="Time")
);

interactive(children=(Dropdown(description='Route', index=1, options=('IV bolus', 'Oral'), value='Oral'), Floa…

## 14. Observation Task 5: Complete PK/PD Assessment

Use the integrated simulation to complete the following tasks.

### Task A: Standard oral dosing

Settings:

- Route = Oral
- Dose = 500 mg
- Vd = 50 L
- CL = 5 L/h
- ka = 1.2 1/h
- F = 0.8
- MEC = 2 mg/L
- MTC = 12 mg/L
- Emax = 100
- EC50 = 5 mg/L
- Hill = 1.5

Record:

- Cmax
- Tmax
- AUC
- Half-life
- Maximum effect
- Average effect
- Time within the therapeutic window

### Task B: Slower oral absorption

Change only ka to 0.3 1/h.

Observe:

- Is Tmax delayed?
- Does Cmax decrease?
- Does AUC change substantially?
- Is the onset of drug effect delayed?

Think about this:

> If a sustained-release formulation is absorbed more slowly, how might its Cmax, Tmax, and duration of effect change?

### Task C: Decreased bioavailability

Change only F to 0.4.

Observe:

- Does Cmax decrease?
- Does AUC decrease?
- Does Average effect decrease?
- Does Time below MEC increase?

Think about this:

> If a patient has poor absorption, or if food/drug interactions reduce oral bioavailability, what therapeutic problems might occur?

### Task D: Decreased clearance

Restore F to 0.8 and change only CL to 2 L/h.

Observe:

- Does AUC increase?
- Is half-life prolonged?
- Does Time above MTC increase?
- Does this suggest increased toxicity risk?

Think about this:

> For patients with decreased renal function or decreased hepatic function, why might the dosing regimen need adjustment even when the dose is unchanged?

## 15. Self-Assessment: Dose, Concentration, and Drug Effect

Complete the following self-assessment based on the content of this notebook. It is recommended that you answer independently first, and then check the reference answers in the next cell.

---

### Question 1: Does increasing the dose always lead to better clinical benefit?

A drug is described by a one-compartment model. After Dose is increased, the plasma concentration increases. Which statement is the most reasonable?

A. The larger the dose, the better the efficacy, with no change in safety.  
B. Increasing the dose may improve efficacy, but it may also increase toxicity risk.  
C. As long as Cmax increases, AUC must decrease.  
D. As long as the drug effect is close to Emax, continuing to increase the dose will still produce a proportional increase in efficacy.  

---

### Question 2: In oral dosing, what is most directly affected by a decrease in bioavailability F?

A. It increases the fraction of drug entering systemic circulation.  
B. It decreases the AUC after oral dosing.  
C. It necessarily increases drug clearance CL.  
D. It necessarily shortens half-life.  

---

### Question 3: When clearance CL decreases, which change is most likely to occur?

A. AUC decreases.  
B. Half-life becomes shorter.  
C. Plasma concentration declines faster.  
D. AUC increases and half-life becomes longer.  

---

### Question 4: Which statement about the Emax model is correct?

A. Concentration and effect are always linearly related.  
B. A larger EC50 usually means that a higher concentration is needed to achieve the same effect.  
C. A smaller Emax means a higher maximum drug effect.  
D. The Hill coefficient is unrelated to the shape of the concentration-effect curve.  

---

### Question 5: When the oral absorption rate constant ka becomes smaller, what is most likely to occur?

A. Tmax is delayed and the concentration rises more slowly.  
B. Tmax occurs earlier and Cmax must increase substantially.  
C. Bioavailability F must become 1.  
D. Clearance CL must decrease.

## 16. Reference Answers for the Self-Assessment

### Question 1

**Reference answer: B**

**Explanation:**  
Increasing the dose usually increases plasma concentration and AUC. If the original concentration is below the effective range, increasing the dose may improve efficacy. However, if the concentration exceeds MTC, toxicity risk may increase. According to the Emax model, once the effect approaches Emax, the additional effect gained by further increasing concentration may be limited.

---

### Question 2

**Reference answer: B**

**Explanation:**  
After oral dosing:

$$
AUC = \frac{F \cdot Dose}{CL}
$$

When Dose and CL remain unchanged, a decrease in F reduces the amount of drug that reaches the systemic circulation, and therefore AUC decreases. F itself does not directly determine CL or half-life.

---

### Question 3

**Reference answer: D**

**Explanation:**  
In the one-compartment model:

$$
k = \frac{CL}{V_d}
$$

$$
t_{1/2} = \frac{0.693 \times V_d}{CL}
$$

When CL decreases, the elimination rate constant k decreases, drug elimination becomes slower, and half-life becomes longer. At the same time:

$$
AUC = \frac{Dose}{CL}
$$

For oral dosing:

$$
AUC = \frac{F \cdot Dose}{CL}
$$

Therefore, a decrease in CL leads to an increase in AUC.

---

### Question 4

**Reference answer: B**

**Explanation:**  
The Emax model describes a nonlinear concentration-effect relationship:

$$
Effect = E_0 + \frac{E_{max} \cdot C^\gamma}{EC_{50}^{\gamma} + C^\gamma}
$$

EC50 is the concentration required to achieve 50% of the maximum effect. A larger EC50 usually means that a higher concentration is needed to achieve the same effect. Emax determines the maximum effect, and the Hill coefficient affects the steepness of the curve.

---

### Question 5

**Reference answer: A**

**Explanation:**  
ka describes the absorption rate of an oral drug. When ka is smaller, the drug is absorbed more slowly, the concentration rises more slowly, Tmax is usually delayed, and Cmax may decrease or the curve may become flatter. ka is not the same as F and does not directly determine CL.

## Notebook Summary

This notebook introduced core PK/PD concepts using a one-compartment model.

Key takeaways:

1. IV bolus dosing usually produces the highest concentration immediately, followed by exponential decline.
2. Oral dosing involves absorption and elimination, so concentration typically rises first and then falls.
3. F, ka, Vd, and CL jointly determine concentration-time profiles, Cmax, Tmax, AUC, and half-life.
4. MEC and MTC provide a simple framework for judging insufficient efficacy and toxicity risk.
5. The Emax model illustrates that drug effect often increases nonlinearly with concentration.
6. Dose optimization aims to balance efficacy and safety by connecting route, dose, exposure, and response.

The complete logic of this section can be summarized as:

$$
Route + Dose \rightarrow Concentration(t) \rightarrow Exposure \rightarrow Effect \rightarrow Efficacy/Safety
$$
